# Phase 3 — Hierarchical LLM Rerank + Learned Self-Check

**Goal**: Close the MRR@20 gap (0.58 → 0.70+). Phase 2 already beat paper on 5/6 metrics. Paper leads on MRR@20 because their LLM hierarchical pipeline recovers queries that retrieval entirely misses. We replicate their recipe but on our **pre-filtered top-100 candidates** with **open-source Llama-3.3-70B** + **learned self-checking**.

**Additions over Phase 2**:
1. **Retrieve top-100** (not 50) — larger recall window
2. **Level 1 Category filter** — LLM picks top-5 categories from candidates
3. **Level 2 API scoring + paper-style self-check** (Q1 direct, Q2 indirect)
4. **Learned self-check**: replace paper's heuristic +20/-10 with reranker probability
5. **Level 3 Endpoint ranking** — final top-20 ordering
6. **Score fusion**: final = α×LLM + β×reranker

**Pre-reqs**: Phase 1 + Phase 2 cache present in `/kaggle/working/cache/` OR attached as input dataset:
- `endpoints.pkl`, `test_queries.json`, `corpus_embs.npy`, `hyde_docs.json`

**Runtime**: ~20-25 min (warm cache + Groq calls).

## Cell 1 — Installs verify

In [ ]:
!pip install -q "httpx==0.27.2" "groq==0.11.0" rank_bm25 sentence-transformers
print("installs verified")

## Cell 2 — Imports + config + load cache

In [ ]:
import os, json, random, pickle, time, re, shutil
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Dict, Tuple, Optional
from collections import defaultdict, Counter

import numpy as np
import torch
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

CACHE_DIR = Path("/kaggle/working/cache")
CACHE_DIR.mkdir(exist_ok=True, parents=True)

# If running with dataset attached, copy from /kaggle/input to working cache
INPUT_CANDIDATES = [
    Path("/kaggle/input/phase1-output-paper2/cache"),
    Path("/kaggle/input/phase2-output-paper2/cache"),
    Path("/kaggle/input/phase1_output_paper2/cache"),
]
for inp in INPUT_CANDIDATES:
    if inp.exists():
        for fname in ["endpoints.pkl", "test_queries.json", "corpus_embs.npy",
                      "hyde_docs.json", "bm25.pkl"]:
            src = inp / fname
            dst = CACHE_DIR / fname
            if src.exists() and not dst.exists():
                print(f"Copying {fname} ({src.stat().st_size/1e6:.1f} MB) from {inp.name}...")
                shutil.copy(src, dst)


@dataclass
class Endpoint:
    endpoint_id: str
    category: str
    tool_name: str
    tool_description: str
    endpoint_name: str
    endpoint_description: str
    method: str = ""
    parameters: List[Dict] = field(default_factory=list)

    def text_for_embedding(self) -> str:
        parts = [
            f"Category: {self.category}",
            f"API: {self.tool_name}",
            f"API description: {self.tool_description}",
            f"Endpoint: {self.endpoint_name}",
            f"Endpoint description: {self.endpoint_description}",
        ]
        if self.method:
            parts.append(f"Method: {self.method}")
        if self.parameters:
            param_str = ", ".join(p.get("name", "") for p in self.parameters[:10])
            parts.append(f"Parameters: {param_str}")
        return " | ".join(parts)


with open(CACHE_DIR / "endpoints.pkl", "rb") as f:
    endpoints: List[Endpoint] = pickle.load(f)
with open(CACHE_DIR / "test_queries.json") as f:
    test_queries = json.load(f)
corpus_embs = np.load(CACHE_DIR / "corpus_embs.npy")

hyde_cache_path = CACHE_DIR / "hyde_docs.json"
hyde_docs: Dict[str, str] = {}
if hyde_cache_path.exists():
    with open(hyde_cache_path) as f:
        hyde_docs = json.load(f)
    print(f"Loaded {len(hyde_docs)} HyDE docs")

print(f"Loaded {len(endpoints)} endpoints, {len(test_queries)} queries, embs {corpus_embs.shape}")

## Cell 3 — Groq client

In [ ]:
from kaggle_secrets import UserSecretsClient
GROQ_API_KEY = UserSecretsClient().get_secret("GROQ_API_KEY")

import httpx
from groq import Groq

http_client = httpx.Client(timeout=120.0, follow_redirects=True)
client = Groq(api_key=GROQ_API_KEY, http_client=http_client)

LLM_MODEL = "llama-3.3-70b-versatile"


def llm_call(prompt: str, temperature: float = 0.0, max_tokens: int = 2000, retries: int = 3) -> str:
    for i in range(retries):
        try:
            resp = client.chat.completions.create(
                model=LLM_MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=temperature,
                max_tokens=max_tokens,
            )
            return resp.choices[0].message.content.strip()
        except Exception as e:
            if i == retries - 1:
                print(f"LLM call failed: {e}")
                return ""
            time.sleep(2 ** i)
    return ""

print("Groq ready:", llm_call("say ready in 2 words", max_tokens=10))

## Cell 4 — Rebuild Phase 2 retrievers + reranker

In [ ]:
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder

def tokenize(text: str) -> List[str]:
    return re.findall(r"[a-zA-Z0-9]+", text.lower())

corpus_texts = [e.text_for_embedding() for e in endpoints]

bm25_path = CACHE_DIR / "bm25.pkl"
if bm25_path.exists():
    with open(bm25_path, "rb") as f:
        bm25 = pickle.load(f)
    print("Loaded BM25 from cache")
else:
    print("Building BM25 (2 min)...")
    tokenized_corpus = [tokenize(t) for t in tqdm(corpus_texts)]
    bm25 = BM25Okapi(tokenized_corpus)
    with open(bm25_path, "wb") as f:
        pickle.dump(bm25, f)

print("Loading BGE embedder + reranker...")
embedder = SentenceTransformer("BAAI/bge-large-en-v1.5", device=DEVICE)
reranker = CrossEncoder("BAAI/bge-reranker-v2-m3", max_length=512, device=DEVICE)

corpus_embs_gpu = torch.from_numpy(corpus_embs).to(DEVICE).float()

# Pre-embed HyDE docs if present
hyde_emb_map: Dict[str, torch.Tensor] = {}
if hyde_docs:
    hyde_texts = [hyde_docs.get(tq["query"], tq["query"]) for tq in test_queries]
    hyde_embs = embedder.encode(hyde_texts, batch_size=32, convert_to_tensor=True,
                                normalize_embeddings=True, show_progress_bar=False).to(DEVICE)
    hyde_emb_map = {tq["query"]: hyde_embs[i] for i, tq in enumerate(test_queries)}
    print(f"HyDE embeddings: {hyde_embs.shape}")


def bm25_search(query, top_k=100):
    scores = bm25.get_scores(tokenize(query))
    top_idx = np.argsort(scores)[::-1][:top_k]
    return [(int(i), float(scores[i])) for i in top_idx]

def bge_search(query, top_k=100):
    q_emb = embedder.encode([f"Represent this sentence for searching relevant passages: {query}"],
                            convert_to_tensor=True, normalize_embeddings=True).to(DEVICE)
    scores = (corpus_embs_gpu @ q_emb.T).squeeze(1)
    top_vals, top_idx = torch.topk(scores, k=top_k)
    return list(zip(top_idx.cpu().tolist(), top_vals.cpu().tolist()))

def hyde_search(query, top_k=100):
    if query in hyde_emb_map:
        q_emb = hyde_emb_map[query].unsqueeze(0)
    else:
        return bge_search(query, top_k)
    scores = (corpus_embs_gpu @ q_emb.T).squeeze(1)
    top_vals, top_idx = torch.topk(scores, k=top_k)
    return list(zip(top_idx.cpu().tolist(), top_vals.cpu().tolist()))

def rrf_fuse(results_list, k=60, top_k=100):
    scores = defaultdict(float)
    for results in results_list:
        for rank, (doc_id, _) in enumerate(results):
            scores[doc_id] += 1.0 / (k + rank + 1)
    return sorted(scores.items(), key=lambda x: -x[1])[:top_k]

def triple_fusion(query, top_k=100):
    return rrf_fuse([bm25_search(query, top_k), bge_search(query, top_k),
                     hyde_search(query, top_k)], top_k=top_k)

print("Retrievers + reranker ready")

## Cell 5 — Build top-100 candidate pool per query (with reranker probs)

Per-query: triple fusion top-100 → reranker scores each. Cached to avoid re-computation across Phase 3 ablations.

In [ ]:
candidates_path = CACHE_DIR / "phase3_candidates.pkl"


def build_candidate_pools(test_queries, top_fusion_k=100, rerank_keep=50):
    pools = {}
    for tq in tqdm(test_queries, desc="Building candidate pools"):
        q = tq["query"]
        # Step 1: triple fusion top-100
        fused = triple_fusion(q, top_k=top_fusion_k)
        cand_ids = [c[0] for c in fused]
        # Step 2: reranker scores all 100 pairs
        pairs = [(q, endpoints[cid].text_for_embedding()) for cid in cand_ids]
        rerank_scores = reranker.predict(pairs, batch_size=16, show_progress_bar=False)
        # Normalize reranker logits -> probability via sigmoid
        rerank_probs = 1.0 / (1.0 + np.exp(-np.array(rerank_scores)))
        pool = [(cid, float(rs), float(rp))
                for cid, rs, rp in zip(cand_ids, rerank_scores, rerank_probs)]
        # Sort by reranker score, keep top rerank_keep
        pool.sort(key=lambda x: -x[1])
        pools[q] = pool[:rerank_keep]
    return pools


if candidates_path.exists():
    with open(candidates_path, "rb") as f:
        candidate_pools = pickle.load(f)
    print(f"Loaded {len(candidate_pools)} cached candidate pools")
else:
    candidate_pools = build_candidate_pools(test_queries, top_fusion_k=100, rerank_keep=50)
    with open(candidates_path, "wb") as f:
        pickle.dump(candidate_pools, f)
    print(f"Built and cached {len(candidate_pools)} candidate pools")

# Sanity: show candidates for one query
sample_q = test_queries[0]["query"]
print(f"\nSample query: {sample_q}")
for cid, rs, rp in candidate_pools[sample_q][:5]:
    print(f"  [rerank_prob={rp:.3f}] {endpoints[cid].endpoint_id}")

## Cell 6 — Level 1: Category filtering prompt

LLM sees unique categories in top-50 pool + query → picks top-5 categories. Narrows search space.

In [ ]:
CAT_PROMPT = """You are an API service discovery expert.

User query: "{query}"

Candidate categories (with counts of how many candidate endpoints are in each):
{category_list}

Pick the TOP 5 categories most relevant to the user query. Consider both direct
topical match and indirect support. Return ONLY a JSON array of category names,
no explanation:
["Category1", "Category2", ...]"""


def level1_category_filter(query: str, pool: List[Tuple[int, float, float]],
                           top_n_cats: int = 5) -> List[str]:
    cats = Counter(endpoints[cid].category for cid, _, _ in pool)
    cat_list_str = "\n".join(f"- {c} ({n})" for c, n in cats.most_common())
    prompt = CAT_PROMPT.format(query=query, category_list=cat_list_str)
    out = llm_call(prompt, temperature=0.0, max_tokens=300)
    # Parse JSON array
    try:
        m = re.search(r"\[.*?\]", out, re.DOTALL)
        if m:
            picked = json.loads(m.group(0))
            picked = [c for c in picked if c in cats]
            if picked:
                return picked[:top_n_cats]
    except Exception:
        pass
    # Fallback: top categories by candidate count
    return [c for c, _ in cats.most_common(top_n_cats)]


# Quick test
sample_q = test_queries[0]["query"]
picked_cats = level1_category_filter(sample_q, candidate_pools[sample_q])
print(f"Query: {sample_q}")
print(f"Picked categories: {picked_cats}")

## Cell 7 — Level 2: API-level scoring with paper-style self-check

For each unique API in filtered categories, LLM assigns 0-100 relevance + rationale.
Then self-check: Q1 direct, Q2 indirect → adjust per paper's +20/-10 heuristic.

In [ ]:
API_SCORE_PROMPT = """You are an API service discovery expert.

User query: "{query}"

Candidate APIs (each may have multiple endpoints). Score each on relevance to query.

{api_list}

For EACH API above, output ONE line in this exact format:
API_NAME | SCORE | RATIONALE

SCORE = integer 0-100 (100 = perfect match, 0 = irrelevant).
RATIONALE = one short sentence.
No other text before or after."""


SELF_CHECK_PROMPT = """User query: "{query}"

For each API below, answer two yes/no questions:
Q1: Does this API DIRECTLY support the query?
Q2: Can this API INDIRECTLY assist?

APIs:
{api_list}

Output format, ONE line per API:
API_NAME | Q1_YES_OR_NO | Q2_YES_OR_NO

No other text."""


def level2_api_scoring(query: str, pool: List[Tuple[int, float, float]],
                      categories_keep: List[str], max_apis: int = 15) -> Dict[str, Dict]:
    # Group pool by API (tool_name), keep only those in picked categories
    by_api: Dict[str, List] = defaultdict(list)
    for cid, rs, rp in pool:
        e = endpoints[cid]
        if e.category in categories_keep or not categories_keep:
            by_api[e.tool_name].append((cid, rs, rp, e))
    if not by_api:
        # Fallback to all pool
        for cid, rs, rp in pool:
            e = endpoints[cid]
            by_api[e.tool_name].append((cid, rs, rp, e))

    # Rank APIs by max reranker prob, take top max_apis
    api_order = sorted(by_api.keys(),
                       key=lambda t: -max(ep[2] for ep in by_api[t]))[:max_apis]
    # Build prompt list
    api_list_str = ""
    api_desc_map: Dict[str, str] = {}
    for t in api_order:
        first = by_api[t][0][3]
        desc = (first.tool_description or first.endpoint_description)[:200]
        api_list_str += f"- {t}: {desc}\n"
        api_desc_map[t] = desc

    # LLM scoring call
    out = llm_call(API_SCORE_PROMPT.format(query=query, api_list=api_list_str),
                   temperature=0.2, max_tokens=1500)
    scores: Dict[str, Dict] = {}
    for line in out.splitlines():
        parts = [p.strip() for p in line.split("|")]
        if len(parts) >= 2 and parts[0] in by_api:
            try:
                score = float(re.findall(r"\d+", parts[1])[0])
                scores[parts[0]] = {"llm_score": score, "rationale": parts[2] if len(parts) > 2 else ""}
            except Exception:
                pass
    # Fill missing with default
    for t in api_order:
        if t not in scores:
            scores[t] = {"llm_score": 50.0, "rationale": ""}

    # Paper-style self-check
    sc_out = llm_call(SELF_CHECK_PROMPT.format(query=query, api_list=api_list_str),
                      temperature=0.0, max_tokens=800)
    for line in sc_out.splitlines():
        parts = [p.strip() for p in line.split("|")]
        if len(parts) >= 3 and parts[0] in scores:
            q1_yes = "yes" in parts[1].lower()
            q2_yes = "yes" in parts[2].lower()
            scores[parts[0]]["q1"] = q1_yes
            scores[parts[0]]["q2"] = q2_yes
            adj = (20 if q1_yes else -10) + (10 if q2_yes else 0)
            scores[parts[0]]["sc_adjusted"] = scores[parts[0]]["llm_score"] + adj
    # Default self-check values
    for t, info in scores.items():
        info.setdefault("q1", False)
        info.setdefault("q2", False)
        info.setdefault("sc_adjusted", info["llm_score"])
        info["api_candidates"] = by_api[t]
    return scores


# Sanity test
sample_q = test_queries[0]["query"]
cats = level1_category_filter(sample_q, candidate_pools[sample_q])
api_scores = level2_api_scoring(sample_q, candidate_pools[sample_q], cats)
print(f"Query: {sample_q}")
for api, info in sorted(api_scores.items(), key=lambda x: -x[1]["sc_adjusted"])[:5]:
    print(f"  [LLM={info['llm_score']:.0f}, SC={info['sc_adjusted']:.0f}, "
          f"Q1={info['q1']}, Q2={info['q2']}] {api}")

## Cell 8 — Level 3: Endpoint ranking within top APIs

For top APIs, ask LLM to rank endpoints within each. Output final top-20.

In [ ]:
EP_RANK_PROMPT = """User query: "{query}"

Candidate endpoints (from relevant APIs):
{endpoint_list}

Rank these endpoints by relevance to the user query.
Output ONE line per endpoint in this exact format:
ENDPOINT_ID | SCORE

SCORE = integer 0-100. Include ALL endpoints above. No other text."""


def level3_endpoint_rank(query: str, api_scores: Dict[str, Dict],
                        max_apis_to_rank: int = 10) -> List[Tuple[int, float]]:
    # Gather endpoints from top-K APIs by sc_adjusted
    sorted_apis = sorted(api_scores.items(),
                         key=lambda x: -x[1]["sc_adjusted"])[:max_apis_to_rank]
    cand_list = []
    for api_name, info in sorted_apis:
        for cid, rs, rp, e in info["api_candidates"]:
            cand_list.append((cid, rs, rp, e, info["sc_adjusted"], api_name))

    if not cand_list:
        return []

    # Build endpoint prompt list (tag each with a short ID)
    ep_list_str = ""
    ep_id_map: Dict[str, int] = {}
    for idx, (cid, _, _, e, _, api_name) in enumerate(cand_list):
        short_id = f"E{idx}"
        ep_id_map[short_id] = cid
        desc = e.endpoint_description[:150].replace("\n", " ")
        ep_list_str += f"- {short_id} [{api_name}] {e.endpoint_name}: {desc}\n"

    out = llm_call(EP_RANK_PROMPT.format(query=query, endpoint_list=ep_list_str),
                   temperature=0.1, max_tokens=2000)

    llm_ep_scores: Dict[int, float] = {}
    for line in out.splitlines():
        parts = [p.strip() for p in line.split("|")]
        if len(parts) >= 2 and parts[0] in ep_id_map:
            try:
                score = float(re.findall(r"\d+", parts[1])[0])
                llm_ep_scores[ep_id_map[parts[0]]] = score
            except Exception:
                pass

    # Fill missing with API-level score as prior
    results = []
    for cid, rs, rp, e, api_score, api_name in cand_list:
        ep_score = llm_ep_scores.get(cid, api_score * 0.7)
        results.append((cid, rs, rp, api_score, ep_score))
    return results


# Sanity test
sample_q = test_queries[0]["query"]
cats = level1_category_filter(sample_q, candidate_pools[sample_q])
api_scores = level2_api_scoring(sample_q, candidate_pools[sample_q], cats)
ep_results = level3_endpoint_rank(sample_q, api_scores)
print(f"Query: {sample_q}")
print(f"Got {len(ep_results)} endpoint scores")
for cid, rs, rp, api_s, ep_s in sorted(ep_results, key=lambda x: -x[4])[:5]:
    print(f"  [ep={ep_s:.0f}, api={api_s:.0f}, rerank={rp:.2f}] {endpoints[cid].endpoint_id}")

## Cell 9 — Phase 3 full pipeline: score fusion retrievers

Two variants:
- **P3-heuristic**: paper's +20/-10 self-check
- **P3-learned**: replace heuristic with α×(reranker_prob − 0.5)×100

Final score fusion: `final = w_llm × ep_score + w_rerank × rerank_prob×100`

In [ ]:
def phase3_pipeline(query: str,
                   top_k: int = 20,
                   mode: str = "heuristic",
                   w_llm: float = 0.6,
                   w_rerank: float = 0.4,
                   alpha: float = 40.0) -> List[Tuple[int, float]]:
    """mode: 'heuristic' (paper's +20/-10) or 'learned' (reranker prob-based)."""
    pool = candidate_pools[query]
    # L1 Category
    cats = level1_category_filter(query, pool, top_n_cats=5)
    # L2 API scoring
    api_scores = level2_api_scoring(query, pool, cats, max_apis=15)

    # Swap self-check if learned mode
    if mode == "learned":
        for api_name, info in api_scores.items():
            # Use max reranker_prob among this API's endpoints
            max_prob = max(ep[2] for ep in info["api_candidates"])
            info["sc_adjusted"] = info["llm_score"] + alpha * (max_prob - 0.5)

    # L3 Endpoint ranking
    ep_results = level3_endpoint_rank(query, api_scores, max_apis_to_rank=10)

    # Final score fusion: LLM ep_score + rerank prob
    scored = []
    for cid, rs, rp, api_s, ep_s in ep_results:
        final = w_llm * ep_s + w_rerank * (rp * 100)
        scored.append((cid, final))

    # Fallback: if fewer than top_k in L3 pool, backfill from reranker pool
    seen = {cid for cid, _ in scored}
    for cid, rs, rp in pool:
        if cid not in seen:
            scored.append((cid, rp * 50.0))  # lower prior
            seen.add(cid)

    scored.sort(key=lambda x: -x[1])
    return scored[:top_k]


# Quick sanity
sample_q = test_queries[0]["query"]
top20 = phase3_pipeline(sample_q, top_k=20, mode="heuristic")
print(f"Query: {sample_q}")
print(f"Gold: {test_queries[0]['gold_endpoint_id']}")
for i, (cid, score) in enumerate(top20[:5], 1):
    marker = " <-- GOLD" if endpoints[cid].endpoint_id == test_queries[0]['gold_endpoint_id'] else ""
    print(f"  {i}. [{score:.1f}] {endpoints[cid].endpoint_id}{marker}")

## Cell 10 — Evaluation (across all 50 queries)

Runs 2 Phase 3 variants. Each query = 3 LLM calls × ~2 sec = ~6 sec. 50 queries × 6 = ~5 min per variant.

In [ ]:
def evaluate_phase3(mode: str, test_queries, endpoints, ks=(5, 10, 20)):
    id_to_idx = {e.endpoint_id: i for i, e in enumerate(endpoints)}
    max_k = max(ks)
    hits = {k: 0 for k in ks}
    mrrs = {k: 0.0 for k in ks}
    n = len(test_queries)
    per_query_ranks = []

    for tq in tqdm(test_queries, desc=f"Phase 3 ({mode})"):
        gold_idx = id_to_idx.get(tq["gold_endpoint_id"])
        results = phase3_pipeline(tq["query"], top_k=max_k, mode=mode)
        ranked_ids = [r[0] for r in results]
        rank = None
        for r, doc_id in enumerate(ranked_ids, start=1):
            if doc_id == gold_idx:
                rank = r
                break
        per_query_ranks.append(rank)
        for k in ks:
            if rank is not None and rank <= k:
                hits[k] += 1
                mrrs[k] += 1.0 / rank

    out = {f"Hit@{k}": hits[k] / n for k in ks} | {f"MRR@{k}": mrrs[k] / n for k in ks}
    out["_ranks"] = per_query_ranks
    return out


print("\n[1/2] Phase 3 with HEURISTIC self-check (paper recipe)")
res_p3_heur = evaluate_phase3("heuristic", test_queries, endpoints)
print({k: v for k, v in res_p3_heur.items() if not k.startswith("_")})

print("\n[2/2] Phase 3 with LEARNED self-check (reranker-prob)")
res_p3_learned = evaluate_phase3("learned", test_queries, endpoints)
print({k: v for k, v in res_p3_learned.items() if not k.startswith("_")})

# Save
save_obj = {
    "heuristic": {k: v for k, v in res_p3_heur.items() if not k.startswith("_")},
    "learned": {k: v for k, v in res_p3_learned.items() if not k.startswith("_")},
    "ranks_heuristic": res_p3_heur["_ranks"],
    "ranks_learned": res_p3_learned["_ranks"],
}
with open(CACHE_DIR / "phase3_results.json", "w") as f:
    json.dump(save_obj, f, indent=2)

## Cell 11 — Final comparison: Paper vs Phase 1/2/3

In [ ]:
paper_sbert = {"Hit@5": 0.48, "MRR@5": 0.3503, "Hit@10": 0.56, "MRR@10": 0.3609,
               "Hit@20": 0.64, "MRR@20": 0.3668}
paper_gpt4  = {"Hit@5": 0.60, "MRR@5": 0.5073, "Hit@10": 0.66, "MRR@10": 0.5673,
               "Hit@20": 0.74, "MRR@20": 0.6773}

# Try to load earlier phase results for unified table
try:
    with open(CACHE_DIR / "phase1_results.json") as f:
        p1 = json.load(f)
    p1_hybrid = p1.get("hybrid", {})
except Exception:
    p1_hybrid = {}

try:
    with open(CACHE_DIR / "phase2_results.json") as f:
        p2 = json.load(f)
    p2_triple_rerank = p2.get("triple_fusion_rerank", {})
except Exception:
    p2_triple_rerank = {}

p3_heur = {k: v for k, v in res_p3_heur.items() if not k.startswith("_")}
p3_learned = {k: v for k, v in res_p3_learned.items() if not k.startswith("_")}

rows = [
    ("Paper SBERT", paper_sbert),
    ("Paper GPT-4 hierarchical (target)", paper_gpt4),
    ("P1: Hybrid (BM25+BGE)", p1_hybrid),
    ("P2: Triple + Rerank (FULL)", p2_triple_rerank),
    ("P3: +LLM hier. (heuristic SC)", p3_heur),
    ("P3: +LLM hier. (LEARNED SC)", p3_learned),
]

header = f"{'Method':<38} {'Hit@5':<8} {'MRR@5':<8} {'Hit@10':<8} {'MRR@10':<8} {'Hit@20':<8} {'MRR@20':<8}"
print("\n===== FULL PAPER COMPARISON =====")
print(header)
print("-" * len(header))
for name, r in rows:
    if not r:
        print(f"{name:<38} (no data)"); continue
    print(f"{name:<38} "
          f"{r.get('Hit@5', 0):<8.4f} {r.get('MRR@5', 0):<8.4f} "
          f"{r.get('Hit@10', 0):<8.4f} {r.get('MRR@10', 0):<8.4f} "
          f"{r.get('Hit@20', 0):<8.4f} {r.get('MRR@20', 0):<8.4f}")

print("\n===== DELTA (P3 LEARNED vs Paper GPT-4) =====")
for k in ["Hit@5", "MRR@5", "Hit@10", "MRR@10", "Hit@20", "MRR@20"]:
    d = p3_learned.get(k, 0) - paper_gpt4[k]
    sign = "+" if d >= 0 else ""
    print(f"  {k}: ours={p3_learned.get(k, 0):.4f} vs paper={paper_gpt4[k]:.4f}  ({sign}{d:.4f})")

print("\nPhase 3 done.")